# MIE 402 - Pre-Lab 2: Single Pendulum\n**Fall 2026 | 20 points | Data analysis, no coding**\n\nLab 2 uses high-speed video to measure a single pendulum. You will track the bob, convert its position to angle, and compare motion at several initial angles with a theoretical model.\n\n## What you must submit\nSubmit **one Word or PDF file** on Canvas before your own laboratory section begins. Include your name, section, GTA, date, calculations, prediction tables, requested figures, and written answers. You may export this completed notebook or assemble the material in Word. Do not submit blank response cells.\n\n## How to use this notebook\nWork top to bottom. Complete each prediction before running the analysis cell below it. The code is supplied. You do not need to write, repair, or change Python code. Keep all figures visible. Pre-labs normally appear on Canvas on the Monday before the laboratory; the due time is before your own section begins.\n

## 1. The physical system\nThe pivot is fixed. The bob center is a distance $L$ from the pivot. Angle $\theta(t)$ is measured from the downward vertical equilibrium position.\n$$\ddot{\theta}+\frac{g}{L}\sin(\theta)=0.$$\nFor small angles in radians, $\sin(\theta)\approx\theta$:\n$$\omega_n=\sqrt{g/L},\quad f_n=\omega_n/(2\pi),\quad T_0=2\pi\sqrt{L/g}.$$\n\n**Task 1.** For $L=0.82$ m and $g=9.81$ m/s$^2$, calculate $\omega_n$ (rad/s), $f_n$ (Hz), and $T_0$ (s). Does bob mass appear in the ideal result?\n

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\ng=9.81;L=0.82\nomega_n=np.sqrt(g/L);f_n=omega_n/(2*np.pi);T0=2*np.pi/omega_n\nprint(f'omega_n = {omega_n:.3f} rad/s')\nprint(f'f_n = {f_n:.3f} Hz')\nprint(f'T0 = {T0:.3f} s')\n

## 2. Predict before running the simulation\nThe planned initial angles are 8, 35, and 105 degrees. Every trial starts from rest.\n\n**Task 2.** Before running the next cell, complete this table in your submitted file.\n\n| Initial angle | Is small-angle theory appropriate? | Period compared with $T_0$ | Maximum speed compared with 8 degrees |\n|---|---|---|---|\n| 8 degrees | | | |\n| 35 degrees | | | |\n| 105 degrees | | | |\n

In [ ]:
# Supplied nonlinear simulation. Do not modify.\ndef simulate(theta0_deg,dt=0.001,duration=16):\n n=int(duration/dt)+1;t=np.arange(n)*dt;theta=np.zeros(n);omega=np.zeros(n);theta[0]=np.deg2rad(theta0_deg)\n def rhs(q):return np.array([q[1],-(g/L)*np.sin(q[0])])\n q=np.array([theta[0],0.0])\n for i in range(n-1):\n  k1=rhs(q);k2=rhs(q+dt*k1/2);k3=rhs(q+dt*k2/2);k4=rhs(q+dt*k3)\n  q=q+dt*(k1+2*k2+2*k3+k4)/6;theta[i+1],omega[i+1]=q\n return t,theta,omega\nfig,ax=plt.subplots(3,1,figsize=(9,8),sharex=True)\nfor angle,axis in zip([8,35,105],ax):\n t,th,w=simulate(angle);axis.plot(t,np.rad2deg(th));axis.grid(True);axis.set_ylabel('theta (deg)');axis.set_title(f'theta(0) = {angle} deg')\nax[-1].set_xlabel('time (s)');plt.tight_layout()\n

**Task 3.** Estimate one period for each case from the plots. Make a table with your estimates and $T_0$. Which case agrees most closely with small-angle theory? Why does the 105-degree case take longer even though $L$ has not changed?\n\n## 3. From camera coordinates to angle\nThe camera does not measure angle directly. The tracking app returns pivot location $(x_p,y_p)$ and bob location $(x_b,y_b)$ for each frame. Use:\n$$\theta=\operatorname{atan2}(x_b-x_p,\;y_p-y_b).$$\nThe atan2 function preserves the quadrant.\n\n**Task 4.** Explain why a fixed pivot location and good pixel calibration matter. What happens if the bob leaves the video frame?\n

In [ ]:
t,theta,omega=simulate(35,duration=8)\nxp,yp,Lmm=500.,120.,820.\nxb=xp+Lmm*np.sin(theta);yb=yp+Lmm*np.cos(theta)\ntheta_xy=np.arctan2(xb-xp,yp-yb)\nfig,ax=plt.subplots(1,2,figsize=(11,4))\nax[0].plot(xb,yb);ax[0].plot(xp,yp,'ro',label='pivot');ax[0].set_aspect('equal');ax[0].invert_yaxis();ax[0].set_xlabel('x (mm)');ax[0].set_ylabel('y (mm)');ax[0].legend();ax[0].set_title('Tracked bob path')\nax[1].plot(t,np.rad2deg(theta_xy));ax[1].grid(True);ax[1].set_xlabel('time (s)');ax[1].set_ylabel('theta (deg)');ax[1].set_title('Angle recovered from x and y');plt.tight_layout()\n

## 4. Frame rate and frequency\nThe camera records at 1000 frames/s. The pendulum frequency is near 0.55 Hz, so each cycle contains many frames. High frame rate also improves numerical angular-velocity estimates.\n\n**Task 5.** Calculate the Nyquist frequency for 1000 frames/s. Approximately how many frames occur in one small-angle period? Explain why actual frame rate belongs in lab notes.\n

In [ ]:
camera_fs=1000\nprint(f'Camera Nyquist frequency = {camera_fs/2:.0f} Hz')\nprint(f'Frames in one small-angle period = {camera_fs*T0:.0f}')\nt,theta,omega=simulate(35,duration=16);dt=t[1]-t[0];N=len(theta)\nP=np.abs(np.fft.rfft(theta-theta.mean()))/N;P[1:-1]*=2;f=np.fft.rfftfreq(N,dt)\nplt.figure(figsize=(8,4));plt.plot(f,P);plt.xlim(0,3);plt.grid(True);plt.xlabel('frequency (Hz)');plt.ylabel('one-sided amplitude (rad)');plt.title('Simulated 35-degree pendulum spectrum')\nprint(f'FFT-bin spacing = {1/(N*dt):.4f} Hz')\n

## 5. Plan the laboratory record\n**Task 6.** Write a short plan covering: how the group sets and records each initial angle; camera settings checked before recording; quantities recorded from the tracking result; and one data-quality check before leaving.\n\nMention all three Lab 2 ranges: below 10 degrees, 10 to 90 degrees, and 90 to 180 degrees. State that every release starts from rest.\n